# Semantic Drift: Entity-Level and Overall Semantic Fidelity


## Data Processing & Version Alignment

This section loads the seven paraphrased CSV datasets, normalizes version names, pivots them into a paired format, and prints the shape and sample rows for each dataset.

In [0]:
import pandas as pd
from pathlib import Path
import spacy
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display for easier debugging and inspection
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

# List of all paraphrased dataset CSV files
DATASETS = [
    "cmv_paraphrased.csv",
    "eli5_paraphrased.csv",
    "sci_gen_paraphrased.csv",
    "tldr_paraphrased.csv",
    "wp_paraphrased.csv",
    "xsum_paraphrased.csv",
    "yelp_paraphrased.csv",
]

# Directory containing the CSV files
DATA_DIR = Path("data")

# Cached pickle file to avoid rebuilding the paired dataset every run
PAIRED_ALL_PICKLE = Path("paired_all_t1.pkl")


# Load all CSVs, normalize version names, pivot into paired format, and cache result
def load_or_build_dataset():
    # If cached file exists, load it instead of recomputing
    if PAIRED_ALL_PICKLE.exists():
        print("Loading cached paired_all_t1.pkl ...")
        return pd.read_pickle(PAIRED_ALL_PICKLE)

    print("Building paired_all_t1 dataset from CSV files...")

    dfs = []
    for fname in DATASETS:
        # Load each dataset
        df = pd.read_csv(DATA_DIR / fname)

        # Add dataset name (e.g., "cmv", "eli5")
        df["dataset"] = fname.replace("_paraphrased.csv", "")

        # Fix common typo in version_name column
        df["version_name"] = df["version_name"].replace("orignal", "original")

        dfs.append(df)

    # Combine all datasets into one large dataframe
    full = pd.concat(dfs, ignore_index=True)

    # Map version names to the column names we want in the pivoted table
    target_versions = {
        "original": "text_T0",
        "chatgpt": "text_chatgpt",
        "dipper(low)": "text_dipper_low",
        "dipper": "text_dipper",
        "dipper(high)": "text_dipper_high",
        "pegasus(slight)": "text_pegasus_slight",
        "pegasus(full)": "text_pegasus_full",
        "palm": "text_palm",
    }

    # Keep only rows with a recognized version
    filtered = full.copy()
    filtered["version_col"] = filtered["version_name"].map(target_versions)
    filtered = filtered[filtered["version_col"].notna()].copy()

    # Pivot so each row contains all paraphrased versions of the same article
    paired = filtered.pivot(
        index=["dataset", "key", "source"],
        columns="version_col",
        values="text"
    ).reset_index()

    # Reorder columns for readability
    paired = paired[[
        "dataset", "key", "source",
        "text_T0",
        "text_chatgpt",
        "text_palm",
        "text_dipper_low",
        "text_dipper",
        "text_dipper_high",
        "text_pegasus_slight",
        "text_pegasus_full",
    ]]

    # Cache the result for faster future runs
    paired.to_pickle(PAIRED_ALL_PICKLE)
    print("Saved paired_all_t1.pkl")

    return paired


In [0]:
# Load the paired dataset (cached if available)
paired_t1 = load_or_build_dataset()

# Print overall shape of the paired dataframe
print("paired_all_t1 shape:", paired_t1.shape)

# Show two sample rows from each dataset to verify structure
for ds in paired_t1["dataset"].unique():
    print(f"\n--- {ds} ---")
    display(paired_t1[paired_t1["dataset"] == ds].head(2))


## NER Drift Studies across All Datasets

In [0]:
NER_SETS_PICKLE = Path("paired_ner_t1.pkl")
NER_METRICS_PICKLE = Path("ner_metrics_t1.pkl")

def get_ner_set(text, nlp):
    if not isinstance(text, str) or text.strip() == "":
        return set()
    doc = nlp(text)
    return {ent.text.lower().strip() for ent in doc.ents if ent.text.strip() != ""}

def load_or_compute_ner_sets(paired):
    if NER_SETS_PICKLE.exists():
        print("Loading cached paired_ner_t1.pkl ...")
        return pd.read_pickle(NER_SETS_PICKLE)

    print("Computing NER sets for T1 dataset...")
    nlp = spacy.load("en_core_web_sm")

    df = paired.copy()

    df["ents_T0"]        = df["text_T0"].apply(lambda x: get_ner_set(x, nlp))
    df["ents_chatgpt"]   = df["text_chatgpt"].apply(lambda x: get_ner_set(x, nlp))
    df["ents_palm"]      = df["text_palm"].apply(lambda x: get_ner_set(x, nlp))
    df["ents_dip_low"]   = df["text_dipper_low"].apply(lambda x: get_ner_set(x, nlp))
    df["ents_dip_mid"]   = df["text_dipper"].apply(lambda x: get_ner_set(x, nlp))
    df["ents_dip_high"]  = df["text_dipper_high"].apply(lambda x: get_ner_set(x, nlp))
    df["ents_pg_slight"] = df["text_pegasus_slight"].apply(lambda x: get_ner_set(x, nlp))
    df["ents_pg_full"]   = df["text_pegasus_full"].apply(lambda x: get_ner_set(x, nlp))

    df.to_pickle(NER_SETS_PICKLE)
    print("Saved paired_ner_t1.pkl")
    return df

paired_ner = load_or_compute_ner_sets(paired_t1)


In [0]:
def ner_metrics(A, B):
    if len(A) == 0:
        return np.nan, np.nan, np.nan

    inter = len(A & B)
    union = len(A | B)

    j = inter / union if union > 0 else 0
    r = inter / len(A)
    p = inter / len(B) if len(B) > 0 else np.nan

    return j, r, p

def load_or_compute_ner_metrics(paired_ner):
    if NER_METRICS_PICKLE.exists():
        print("Loading cached ner_metrics_t1.pkl ...")
        return pd.read_pickle(NER_METRICS_PICKLE)

    print("Computing NER metrics for T1 dataset...")

    rows = []

    for _, row in paired_ner.iterrows():
        A = row["ents_T0"]

        for name, B in [
            ("chatgpt",        row["ents_chatgpt"]),
            ("palm",           row["ents_palm"]),
            ("dipper_low",     row["ents_dip_low"]),
            ("dipper_mid",     row["ents_dip_mid"]),
            ("dipper_high",    row["ents_dip_high"]),
            ("pegasus_slight", row["ents_pg_slight"]),
            ("pegasus_full",   row["ents_pg_full"]),
        ]:
            j, r, p = ner_metrics(A, B)
            rows.append({
                "dataset": row["dataset"],
                "key": row["key"],
                "source": row["source"],
                "paraphraser": name,
                "jaccard": j,
                "recall": r,
                "precision": p,
                "has_entities_T0": len(A) > 0,
            })

    ner_df = pd.DataFrame(rows)
    ner_df.to_pickle(NER_METRICS_PICKLE)
    print("Saved ner_metrics_t1.pkl")

    return ner_df

ner_df = load_or_compute_ner_metrics(paired_ner)


In [0]:
# Keep only rows where the original text (T0) actually had entities
valid = ner_df[ner_df["has_entities_T0"]]
excluded = len(ner_df) - len(valid)

print(f"Excluded articles (no T0 entities): {excluded} ({excluded/len(ner_df):.2%})")

# Desired paraphraser order
paraphraser_order = [
    "chatgpt",
    "palm",
    "dipper_low",
    "dipper_mid",
    "dipper_high",
    "pegasus_slight",
    "pegasus_full",
]

# Compute mean and std for Jaccard, Recall, Precision for each paraphraser
summary = (
    valid
    .groupby("paraphraser")[["jaccard", "recall", "precision"]]
    .agg(["mean", "std"])
    .reindex(paraphraser_order)   # enforce order
)
display(summary)

# Compute mean and std for recall and precision
global_metrics = (
    valid
    .groupby("paraphraser")[["recall", "precision"]]
    .agg(["mean", "std"])
    .reindex(paraphraser_order)   # enforce order
)
display(global_metrics)

# Build a clean long-format dataframe
rows = []
for paraphraser in paraphraser_order:
    for metric in ["recall", "precision"]:
        mean_val = global_metrics.loc[paraphraser, (metric, "mean")]
        std_val = global_metrics.loc[paraphraser, (metric, "std")]
        rows.append([paraphraser, metric, mean_val, std_val])

plot_df = pd.DataFrame(rows, columns=["paraphraser", "metric", "value_mean", "value_std"])

# ---- Grouped bar chart with correct error bars ----
plt.figure(figsize=(9.5, 6.5))

sns.barplot(
    data=plot_df,
    x="paraphraser",
    y="value_mean",
    hue="metric",
    order=paraphraser_order,   # enforce order
    errorbar=None,
    capsize=0.15
)

# Add manual error bars
ax = plt.gca()
num_paraphrasers = len(paraphraser_order)
bar_width = 0.8 / 2

for i, row in plot_df.iterrows():
    paraphraser_index = paraphraser_order.index(row["paraphraser"])
    metric_offset = -bar_width/2 if row["metric"] == "recall" else bar_width/2

    x_loc = paraphraser_index + metric_offset

    plt.errorbar(
        x=x_loc,
        y=row["value_mean"],
        yerr=row["value_std"],
        fmt="none",
        c="black",
        capsize=5
    )

plt.title("Global Recall and Precision by Paraphraser", fontsize=18)
plt.xlabel("Paraphraser", fontsize=16)
plt.ylabel("Mean", fontsize=16)

plt.xticks(
    range(num_paraphrasers),
    paraphraser_order,
    rotation=45,
    ha="right",
    fontsize=14
)

plt.yticks(fontsize=14)

plt.legend(
    title="Metric",
    fontsize=11,
    title_fontsize=12
)

plt.tight_layout()

plt.savefig("output/ner_global_recall_precision_t1.png", dpi=300, bbox_inches="tight")

plt.show()


In [0]:
# Filter only Pegasus paraphrasers for comparison
pg = valid[valid["paraphraser"].isin(["pegasus_slight", "pegasus_full"])]

# Compute mean and std for Jaccard, Recall, Precision
pg_summary = pg.groupby("paraphraser")[["jaccard", "recall", "precision"]].agg(["mean", "std"])
display(pg_summary)

# Metrics to compare
metrics = ["jaccard", "recall", "precision"]
x = np.arange(len(metrics))

bar_width = 0.25   # narrower bars

# Extract means and stds for plotting
means_s = [pg_summary.loc["pegasus_slight"][(m, "mean")] for m in metrics]
means_f = [pg_summary.loc["pegasus_full"][(m, "mean")] for m in metrics]

std_s = [pg_summary.loc["pegasus_slight"][(m, "std")] for m in metrics]
std_f = [pg_summary.loc["pegasus_full"][(m, "std")] for m in metrics]

# Plot side‑by‑side comparison
fig, ax = plt.subplots(figsize=(8, 5))

# Bars for Pegasus slight
ax.bar(
    x - bar_width/2,
    means_s,
    width=bar_width,
    label="Pegasus (slight)",
    alpha=0.9
)
ax.errorbar(
    x - bar_width/2,
    means_s,
    yerr=std_s,
    fmt="none",
    ecolor="black",
    capsize=5,
    linewidth=1.2
)

# Bars for Pegasus full
ax.bar(
    x + bar_width/2,
    means_f,
    width=bar_width,
    label="Pegasus (full)",
    alpha=0.9
)
ax.errorbar(
    x + bar_width/2,
    means_f,
    yerr=std_f,
    fmt="none",
    ecolor="black",
    capsize=5,
    linewidth=1.2
)

ax.set_xticks(x)
ax.set_xticklabels(["Jaccard", "Recall", "Precision"], fontsize=12)
ax.set_ylabel("Score", fontsize=14)
ax.set_title("Entity Stability: Pegasus(slight) vs Pegasus(full)", fontsize=16)
ax.legend(fontsize=12)

plt.tight_layout()

# Save to output directory
plt.savefig("output/ner_pegasus_comparison_t1.png", dpi=300, bbox_inches="tight")

plt.show()


In [0]:
# Metrics to analyze at the domain level
metrics = ["jaccard", "recall", "precision"]

# Create a 1×3 subplot layout (one plot per metric)
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True)

for ax, metric in zip(axes, metrics):

    # Compute mean and std for each dataset × paraphraser combination
    metric_stats = (
        pg
        .groupby(["dataset", "paraphraser"])[metric]
        .agg(["mean", "std"])
        .reset_index()
    )

    # Plot bar chart with error bars
    sns.barplot(
        data=pg,
        x="dataset",
        y=metric,
        hue="paraphraser",
        ax=ax,
        errorbar="sd",
        capsize=0.15
    )

    ax.set_title(f"{metric.capitalize()}", fontsize=16)
    ax.set_ylabel(f"Mean {metric.capitalize()}", fontsize=14)
    ax.set_xlabel("Dataset", fontsize=14)
    ax.tick_params(axis="x", rotation=45, labelsize=12)
    ax.tick_params(axis="y", labelsize=12)

    # ---- Legend bottom-right ----
    ax.legend(
        title="Paraphraser",
        fontsize=11,
        title_fontsize=12,
        loc="lower right"
    )

plt.tight_layout()

# ---- SAVE 3 INDIVIDUAL PLOTS ----
for metric in metrics:
    fig_single, ax_single = plt.subplots(figsize=(7, 5))

    sns.barplot(
        data=pg,
        x="dataset",
        y=metric,
        hue="paraphraser",
        ax=ax_single,
        errorbar="sd",
        capsize=0.15
    )

    ax_single.set_title(f"{metric.capitalize()}", fontsize=16)
    ax_single.set_ylabel(f"Mean {metric.capitalize()}", fontsize=14)
    ax_single.set_xlabel("Dataset", fontsize=14)
    ax_single.tick_params(axis="x", rotation=45, labelsize=12)
    ax_single.tick_params(axis="y", labelsize=12)

    # ---- Legend bottom-right ----
    ax_single.legend(
        title="Paraphraser",
        fontsize=11,
        title_fontsize=12,
        loc="lower right"
    )

    plt.tight_layout()
    plt.savefig(f"output/ner_by_dataset_peg_{metric}_t1.png", dpi=300, bbox_inches="tight")
    plt.close(fig_single)

plt.show()


In [0]:
# Filter only Dipper paraphrasers (low, mid, high intensity)
dipper_only = ner_df[ner_df["paraphraser"].isin([
    "dipper_low", "dipper_mid", "dipper_high"
])]

# Compute mean and std for recall + precision
stats = (
    dipper_only
    .groupby("paraphraser")[["recall", "precision"]]
    .agg(["mean", "std"])
    .reindex(["dipper_low", "dipper_mid", "dipper_high"])
)

paraphrasers = ["dipper_low", "dipper_mid", "dipper_high"]
metrics = ["recall", "precision"]

x = np.arange(len(paraphrasers))
bar_width = 0.25   # narrower bars

plt.figure(figsize=(8, 5))

# Plot bars + error bars manually
for i, metric in enumerate(metrics):
    means = stats[(metric, "mean")].values
    stds  = stats[(metric, "std")].values

    # Offset bars for recall vs precision
    offset = (i - 0.5) * bar_width

    # Bars
    plt.bar(
        x + offset,
        means,
        width=bar_width,
        label=metric.capitalize(),
        alpha=0.9
    )

    # Classic Matplotlib error bars
    plt.errorbar(
        x + offset,
        means,
        yerr=stds,
        fmt="none",
        ecolor="black",
        capsize=5,
        linewidth=1.2
    )

plt.xticks(x, ["dipper_low", "dipper_mid", "dipper_high"], fontsize=12)
plt.ylabel("Score (Higher = Better Entity Preservation)", fontsize=14)
plt.xlabel("Dipper Intensity", fontsize=14)
plt.title("NER Recall and Precision vs Dipper Intensity", fontsize=16)
plt.ylim(0, 1)
plt.legend(fontsize=12)

plt.tight_layout()

# Save to output directory
plt.savefig("output/ner_dipper_recall_precision_t1.png", dpi=300, bbox_inches="tight")

plt.show()



In [0]:
# ---------------------------------------------------------
# INDIVIDUAL PLOT: RECALL
# ---------------------------------------------------------

fig_r, ax_r = plt.subplots(figsize=(9, 6))

bar_width = 0.35
x = np.arange(len(paraphraser_order))

# Extract recall means and stds
means_h = rp_stats[rp_stats["source_group"] == "Human"][("recall", "mean")].values
means_l = rp_stats[rp_stats["source_group"] == "LLM"][("recall", "mean")].values
std_h   = rp_stats[rp_stats["source_group"] == "Human"][("recall", "std")].values
std_l   = rp_stats[rp_stats["source_group"] == "LLM"][("recall", "std")].values

# Human bars
ax_r.bar(x - bar_width/2, means_h, width=bar_width, label="Human", alpha=0.85)
ax_r.errorbar(x - bar_width/2, means_h, yerr=std_h, fmt="none", ecolor="black", capsize=4)

# LLM bars
ax_r.bar(x + bar_width/2, means_l, width=bar_width, label="LLM", alpha=0.85)
ax_r.errorbar(x + bar_width/2, means_l, yerr=std_l, fmt="none", ecolor="black", capsize=4)

ax_r.set_title("Recall", fontsize=16)
ax_r.set_xticks(x)
ax_r.set_xticklabels(paraphraser_order, rotation=45, ha="right", fontsize=12)
ax_r.set_ylim(0, 1.1)
ax_r.set_yticks(np.arange(0, 1.1, 0.2))
ax_r.set_ylabel("Score", fontsize=14)
ax_r.legend(title="Source Group", fontsize=11, title_fontsize=12)

plt.tight_layout()
plt.savefig("output/ner_human_vs_llm_recall_t1.png", dpi=300, bbox_inches="tight")
plt.show()


# ---------------------------------------------------------
# INDIVIDUAL PLOT: PRECISION
# ---------------------------------------------------------

fig_p, ax_p = plt.subplots(figsize=(9, 6))

# Extract precision means and stds
means_h = rp_stats[rp_stats["source_group"] == "Human"][("precision", "mean")].values
means_l = rp_stats[rp_stats["source_group"] == "LLM"][("precision", "mean")].values
std_h   = rp_stats[rp_stats["source_group"] == "Human"][("precision", "std")].values
std_l   = rp_stats[rp_stats["source_group"] == "LLM"][("precision", "std")].values

# Human bars
ax_p.bar(x - bar_width/2, means_h, width=bar_width, label="Human", alpha=0.85)
ax_p.errorbar(x - bar_width/2, means_h, yerr=std_h, fmt="none", ecolor="black", capsize=4)

# LLM bars
ax_p.bar(x + bar_width/2, means_l, width=bar_width, label="LLM", alpha=0.85)
ax_p.errorbar(x + bar_width/2, means_l, yerr=std_l, fmt="none", ecolor="black", capsize=4)

ax_p.set_title("Precision", fontsize=16)
ax_p.set_xticks(x)
ax_p.set_xticklabels(paraphraser_order, rotation=45, ha="right", fontsize=12)
ax_p.set_ylim(0, 1.1)
ax_p.set_yticks(np.arange(0, 1.1, 0.2))
ax_p.set_ylabel("Score", fontsize=14)
ax_p.legend(title="Source Group", fontsize=11, title_fontsize=12)

plt.tight_layout()
plt.savefig("output/ner_human_vs_llm_precision_t1.png", dpi=300, bbox_inches="tight")
plt.show()


## BERTScore Studies across All Datasets

In [0]:
# ---------------------------------------------------------
# CHECK TEXT LENGTHS BEFORE CHUNKING
# ---------------------------------------------------------

from transformers import AutoTokenizer
import pandas as pd

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def get_token_length(text):
    return len(tokenizer(text, truncation=False)["input_ids"])

# Compute lengths for all T0 texts (or all columns if you want)
lengths = paired_t1["text_T0"].dropna().astype(str).apply(get_token_length)

print("Token length statistics for text_T0:")
print(lengths.describe())

print("\nExamples of very long texts:")
print(lengths.sort_values(ascending=False).head(10))


In [0]:
all_lengths = []

TEXT_COLUMNS = [
    "text_T0",
    "text_chatgpt",
    "text_palm",
    "text_dipper_low",
    "text_dipper",
    "text_dipper_high",
    "text_pegasus_slight",
    "text_pegasus_full",
]

for col in TEXT_COLUMNS:
    col_lengths = paired_t1[col].dropna().astype(str).apply(get_token_length)
    print(f"\nToken length stats for {col}:")
    print(col_lengths.describe())
    all_lengths.append(col_lengths)

# Combine all into one big series if needed
combined = pd.concat(all_lengths)
print("\nCombined token length stats across all text columns:")
print(combined.describe())


## RoBERTa Embedding Extraction by Dataset


In [0]:
def run_embedding_pipeline(DATASET_NAME="yelp"):
    
    import torch
    from transformers import AutoTokenizer, AutoModel
    import numpy as np
    import pandas as pd
    from pathlib import Path
    from tqdm import tqdm
    import pickle
    
    # ---------------------------------------------------------
    # CONFIG
    # ---------------------------------------------------------
    
    EMB_PICKLE = Path(f"bert_embeddings_{DATASET_NAME}_fp16.pkl")
    
    MODEL_NAME = "roberta-base"
    
    TEXT_COLUMNS = [
        "text_T0",
        "text_chatgpt",
        "text_palm",
        "text_dipper_low",
        "text_dipper",
        "text_dipper_high",
        "text_pegasus_slight",
        "text_pegasus_full",
    ]
    
    # Chunking parameters
    CHUNK_SIZE = 512
    CHUNK_OVERLAP = 256
    STRIDE = CHUNK_SIZE - CHUNK_OVERLAP
    
    # ---------------------------------------------------------
    # FILTER TO ONE DATASET
    # ---------------------------------------------------------
    
    paired_ds = paired_t1[paired_t1["dataset"] == DATASET_NAME].copy()
    print(f"Loaded {len(paired_ds)} rows from dataset '{DATASET_NAME}'")
    
    # Extract unique texts for this dataset only
    def extract_unique_texts(df, cols):
        texts = set()
        for col in cols:
            texts.update(df[col].dropna().astype(str).unique())
        return list(texts)
    
    unique_texts = extract_unique_texts(paired_ds, TEXT_COLUMNS)
    print(f"Found {len(unique_texts)} unique texts in '{DATASET_NAME}'")
    
    text_to_id = {txt: i for i, txt in enumerate(unique_texts)}

    # ---------------------------------------------------------
    # EARLY EXIT IF CACHE COMPLETE (NO WARNINGS)
    # ---------------------------------------------------------

    if EMB_PICKLE.exists():
        with open(EMB_PICKLE, "rb") as f:
            cache_obj = pickle.load(f)
            bert_cache = cache_obj["embeddings"]

        if len(bert_cache) == len(unique_texts):
            print("All embeddings already computed — nothing to do.")
            return
    else:
        bert_cache = {}

    # ---------------------------------------------------------
    # DEVICE + MODEL LOAD
    # ---------------------------------------------------------
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print("Using device:", device)
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()
    
    # ---------------------------------------------------------
    # CHUNKED EMBEDDING FUNCTION (FP16)
    # ---------------------------------------------------------
    
    @torch.no_grad()
    def compute_chunked_embeddings_fp16(text):
        encoded = tokenizer(
            text,
            return_tensors="pt",
            truncation=False,
            add_special_tokens=True,
        )
    
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        seq_len = input_ids.size(0)
    
        chunk_embs = []
    
        # Short text → single chunk
        if seq_len <= CHUNK_SIZE:
            batch = {
                "input_ids": input_ids.unsqueeze(0).to(device),
                "attention_mask": attention_mask.unsqueeze(0).to(device),
            }
            outputs = model(**batch)
            token_embs = outputs.last_hidden_state.squeeze(0)
            chunk_embs.append(token_embs.to("cpu").half().numpy())
            return chunk_embs
    
        # Long text → sliding window
        start = 0
        while start < seq_len:
            end = min(start + CHUNK_SIZE, seq_len)
    
            ids_chunk = input_ids[start:end]
            mask_chunk = attention_mask[start:end]
    
            batch = {
                "input_ids": ids_chunk.unsqueeze(0).to(device),
                "attention_mask": mask_chunk.unsqueeze(0).to(device),
            }
            outputs = model(**batch)
            token_embs = outputs.last_hidden_state.squeeze(0)
    
            chunk_embs.append(token_embs.to("cpu").half().numpy())
    
            if end == seq_len:
                break
    
            start += STRIDE
    
        return chunk_embs
    
    # ---------------------------------------------------------
    # MAIN LOOP — RESUMABLE PER-DATASET CACHE
    # ---------------------------------------------------------
    
    print(f"Starting embedding computation for dataset '{DATASET_NAME}'...")
    
    for text in tqdm(unique_texts):
        tid = text_to_id[text]
    
        # Skip if already computed
        if tid in bert_cache:
            continue
    
        emb_chunks = compute_chunked_embeddings_fp16(text)
        bert_cache[tid] = emb_chunks
    
        # Save every 500 items for safety
        if len(bert_cache) % 500 == 0:
            with open(EMB_PICKLE, "wb") as f:
                pickle.dump(
                    {
                        "embeddings": bert_cache,
                        "text_to_id": text_to_id,
                        "model_name": MODEL_NAME,
                        "dtype": "float16",
                        "chunk_size": CHUNK_SIZE,
                        "chunk_overlap": CHUNK_OVERLAP,
                    },
                    f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )
            print(f"Checkpoint saved ({len(bert_cache)} embeddings).")
    
    # Final save
    with open(EMB_PICKLE, "wb") as f:
        pickle.dump(
            {
                "embeddings": bert_cache,
                "text_to_id": text_to_id,
                "model_name": MODEL_NAME,
                "dtype": "float16",
                "chunk_size": CHUNK_SIZE,
                "chunk_overlap": CHUNK_OVERLAP,
            },
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    
    print(f"Finished dataset '{DATASET_NAME}'. Saved to {EMB_PICKLE}")

run_embedding_pipeline("yelp")


### Inspecting the RoBERTa Embedding Cache File

In [0]:
import pickle
import numpy as np
from transformers import AutoTokenizer

# ---------------------------------------------------------
# LOAD TOKENIZER
# ---------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# ---------------------------------------------------------
# LOAD THE CACHED EMBEDDINGS
# ---------------------------------------------------------

with open("bert_embeddings_cmv_fp16.pkl", "rb") as f:
    obj = pickle.load(f)

embeddings = obj["embeddings"]
text_to_id = obj["text_to_id"]

print("Number of texts:", len(text_to_id))
print("Number of embedding entries:", len(embeddings))

# ---------------------------------------------------------
# SHOW BASIC STRUCTURE
# ---------------------------------------------------------

first_ids = list(embeddings.keys())[:5]
print("First 5 text IDs:", first_ids)

sample_id = first_ids[0]
sample_emb = embeddings[sample_id]

print(f"\nEmbedding entry for text_id {sample_id}:")
print(f"  Number of chunks: {len(sample_emb)}")
print(f"  Shape of first chunk: {sample_emb[0].shape}")

# ---------------------------------------------------------
# SHOW THE CORRESPONDING TEXT
# ---------------------------------------------------------

inv_map = {v: k for k, v in text_to_id.items()}
sample_text = inv_map[sample_id]

print("\nCorresponding text:")
print(sample_text[:500], "...")

# ---------------------------------------------------------
# TOKENIZE AND SHOW FIRST 3 TOKENS
# ---------------------------------------------------------

encoded = tokenizer(sample_text, return_tensors="pt", add_special_tokens=True)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

print("\nFirst 3 tokens:")
print(tokens[:3])

# ---------------------------------------------------------
# PRINT FIRST 20 DIMS OF EMBEDDINGS FOR FIRST 3 TOKENS
# ---------------------------------------------------------

first_chunk = sample_emb[0]   # shape: (num_tokens, 768)

print("\nFirst 3 token embeddings (first 20 dims each):\n")

for i in range(3):
    token_str = tokens[i]
    token_emb = first_chunk[i]

    print(f"Token {i}: {token_str}")
    print("Embedding dims 0–19:", token_emb[:20])
    print()


## Studying Paraphraser Behavior via BERTScore (T1 vs. T0)

### BERTScore Computation

In [0]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import pickle

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

DATASET_NAME = "xsum"   # change to: "cmv", "eli5", "yelp", etc.
EMB_PATH = f"bert_embeddings_{DATASET_NAME}_fp16.pkl"

CHUNK_SIZE = 100   # process 100 rows at a time
BATCH_SIZE = 256   # batching inside BERTScore

cache_path = f"output/bertscore_{DATASET_NAME}_cache.pkl"

# ---------------------------------------------------------
# DEVICE SETUP
# ---------------------------------------------------------

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print("Using device:", device)

# ---------------------------------------------------------
# LOAD EMBEDDINGS
# ---------------------------------------------------------

with open(EMB_PATH, "rb") as f:
    obj = pickle.load(f)

embeddings = obj["embeddings"]
text_to_id = obj["text_to_id"]

print(f"Loaded embeddings for {DATASET_NAME}:")
print("  Number of texts:", len(text_to_id))
print("  Number of embedding entries:", len(embeddings))

# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def flatten_chunks_torch(chunks):
    arr = np.vstack(chunks)
    return torch.tensor(arr, dtype=torch.float32, device=device)

def bertscore_batched_torch(t1_chunks, t0_chunks, batch_size=BATCH_SIZE):
    """
    Memory-safe BERTScore using batching.
    Returns (precision, recall, f1).
    """
    A = flatten_chunks_torch(t1_chunks)
    B = flatten_chunks_torch(t0_chunks)

    A_norm = A / (A.norm(dim=1, keepdim=True) + 1e-8)
    B_norm = B / (B.norm(dim=1, keepdim=True) + 1e-8)

    precision_rows = []
    recall_max = torch.full((B_norm.size(0),), -1e9, device=device)

    for i in range(0, A_norm.size(0), batch_size):
        A_block = A_norm[i:i+batch_size]
        sim_block = A_block @ B_norm.T

        precision_rows.append(sim_block.max(dim=1).values)
        recall_max = torch.maximum(recall_max, sim_block.max(dim=0).values)

    precision = torch.cat(precision_rows).mean().item()
    recall = recall_max.mean().item()
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

# ---------------------------------------------------------
# Load dataset subset
# ---------------------------------------------------------

subset = paired_t1[paired_t1["dataset"] == DATASET_NAME].copy()
subset = subset.reset_index(drop=True)

paraphrasers = [
    "text_chatgpt",
    "text_palm",
    "text_dipper_low",
    "text_dipper",
    "text_dipper_high",
    "text_pegasus_slight",
    "text_pegasus_full",
]

# ---------------------------------------------------------
# Load or initialize cache
# ---------------------------------------------------------

if os.path.exists(cache_path):
    print(f"Resuming from existing cache: {cache_path}")
    with open(cache_path, "rb") as f:
        bert_results = pickle.load(f)
else:
    print("Starting fresh cache...")
    bert_results = {p: [] for p in paraphrasers}

# ---------------------------------------------------------
# Determine starting chunk
# ---------------------------------------------------------

completed = len(next(iter(bert_results.values())))
start_chunk = completed // CHUNK_SIZE

print(f"Already completed {completed} rows. Resuming at chunk {start_chunk}.")

# ---------------------------------------------------------
# Chunked processing loop
# ---------------------------------------------------------

num_rows = len(subset)
num_chunks = (num_rows + CHUNK_SIZE - 1) // CHUNK_SIZE

for chunk_idx in range(start_chunk, num_chunks):
    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, num_rows)
    chunk = subset.iloc[start:end]

    print(f"\nProcessing chunk {chunk_idx+1}/{num_chunks} "
          f"({start} → {end})")

    for _, row in tqdm(chunk.iterrows(), total=len(chunk)):
        t0 = row["text_T0"]
        if t0 not in text_to_id:
            continue

        emb_t0 = embeddings[text_to_id[t0]]

        for p in paraphrasers:
            t1 = row[p]
            if pd.isna(t1) or t1 not in text_to_id:
                bert_results[p].append((None, None, None))
                continue

            emb_t1 = embeddings[text_to_id[t1]]

            prec, rec, f1 = bertscore_batched_torch(emb_t1, emb_t0)
            bert_results[p].append((prec, rec, f1))

    # Save after each chunk
    os.makedirs("output", exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump(bert_results, f)

    print(f"Saved progress to {cache_path}")

print("\nAll chunks completed!")


### Inspecting BERTScore Cache Contents

In [0]:
import pickle
import pandas as pd

cache_path = "output/bertscore_cmv_cache.pkl"

with open(cache_path, "rb") as f:
    bert_results = pickle.load(f)

# High‑level keys
print("Top‑level keys:", list(bert_results.keys()))

# Number of entries per paraphraser
print("\nCounts per paraphraser:")
for p, vals in bert_results.items():
    print(f"  {p}: {len(vals)} entries")

# Peek at first few entries for each paraphraser
print("\nSample entries:")
for p in bert_results:
    print(f"\n--- {p} ---")
    for i, triple in enumerate(bert_results[p][:3]):
        print(f"  {i}: {triple}")


### Plot BERTScore F1

In [0]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------

datasets = ["cmv", "eli5", "sci_gen", "tldr", "wp", "xsum", "yelp"]
paraphrasers = [
    "text_chatgpt",
    "text_palm",
    "text_dipper_low",
    "text_dipper",
    "text_dipper_high",
    "text_pegasus_slight",
    "text_pegasus_full",
]

cache_dir = "output"

# ---------------------------------------------------------
# LOAD AND AGGREGATE GLOBAL MEAN + STD F1
# ---------------------------------------------------------

global_mean_f1 = {}
global_std_f1 = {}

for p in paraphrasers:
    all_f1 = []

    for ds in datasets:
        path = os.path.join(cache_dir, f"bertscore_{ds}_cache.pkl")
        with open(path, "rb") as f:
            bert_results = pickle.load(f)

        f1_vals = [triple[2] for triple in bert_results[p] if triple[2] is not None]
        all_f1.extend(f1_vals)

    global_mean_f1[p] = np.mean(all_f1)
    global_std_f1[p] = np.std(all_f1)

# ---------------------------------------------------------
# BAR PLOT WITH VALUE LABELS
# ---------------------------------------------------------

labels = [p.replace("text_", "") for p in paraphrasers]
means = [global_mean_f1[p] for p in paraphrasers]
stds  = [global_std_f1[p] for p in paraphrasers]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    labels,
    means,
    yerr=stds,
    capsize=4,
    width=0.6,
    color="#1e81b0"
)

plt.ylabel("Global Mean BERTScore F1")
plt.title("Global Mean BERTScore F1 per Paraphraser (Across 7 Datasets)")
plt.xticks(rotation=30)
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Zoom the y-axis to highlight differences
plt.ylim(min(means) - max(stds)*1.2, max(means) + max(stds)*1.2)

# ---------------------------------------------------------
# ADD MEAN VALUE LABELS
# ---------------------------------------------------------
for bar, value in zip(bars, means):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() * 1,
        height + max(stds) * 0.1,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.tight_layout()

plt.savefig("output/global_bertscore_f1.png", dpi=300, bbox_inches="tight")

plt.show()


In [0]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import os

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------

datasets = ["cmv", "eli5", "sci_gen", "tldr", "wp", "xsum", "yelp"]
paraphrasers = [
    "text_chatgpt",
    "text_palm",
    "text_dipper_low",
    "text_dipper",
    "text_dipper_high",
    "text_pegasus_slight",
    "text_pegasus_full",
]

cache_dir = "output"

# ---------------------------------------------------------
# LOAD ALL CACHES
# ---------------------------------------------------------

all_f1 = {}  # dataset → paraphraser → list of f1 scores

for ds in datasets:
    path = os.path.join(cache_dir, f"bertscore_{ds}_cache.pkl")
    print(f"Loading {path} ...")

    with open(path, "rb") as f:
        bert_results = pickle.load(f)

    # Extract F1 scores only
    f1_dict = {}
    for p in paraphrasers:
        f1_dict[p] = [triple[2] for triple in bert_results[p] if triple[2] is not None]

    all_f1[ds] = f1_dict

# ---------------------------------------------------------
# PLOT: 7 SUBPLOTS (TWO ROWS)
# ---------------------------------------------------------

fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharey=True)
axes = axes.flatten()  # flatten to make indexing easier

for i, ds in enumerate(datasets):
    ax = axes[i]
    f1_lists = [all_f1[ds][p] for p in paraphrasers]

    ax.boxplot(
        f1_lists,
        labels=[p.replace("text_", "") for p in paraphrasers],
        showfliers=False
    )
    ax.set_title(ds.upper())
    ax.tick_params(axis="x", rotation=60)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

# Hide the unused 8th subplot
axes[-1].axis("off")

axes[0].set_ylabel("BERTScore F1")
plt.suptitle("BERTScore F1 Across Paraphrasers for 7 Datasets", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])

plt.savefig("output/bertscore_f1_all_datasets_t1.png", dpi=300, bbox_inches="tight")

plt.show()


In [0]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------

datasets = ["cmv", "eli5", "sci_gen", "tldr", "wp", "xsum", "yelp"]
paraphrasers = [
    "text_chatgpt",
    "text_palm",
    "text_dipper_low",
    "text_dipper",
    "text_dipper_high",
    "text_pegasus_slight",
    "text_pegasus_full",
]

cache_dir = "output"

# ---------------------------------------------------------
# LOAD AND AGGREGATE MEAN + STD F1
# ---------------------------------------------------------

mean_f1 = {ds: [] for ds in datasets}
std_f1  = {ds: [] for ds in datasets}

for ds in datasets:
    path = os.path.join(cache_dir, f"bertscore_{ds}_cache.pkl")
    with open(path, "rb") as f:
        bert_results = pickle.load(f)

    for p in paraphrasers:
        f1_vals = [triple[2] for triple in bert_results[p] if triple[2] is not None]
        mean_f1[ds].append(np.mean(f1_vals))
        std_f1[ds].append(np.std(f1_vals))

# ---------------------------------------------------------
# LINE PLOT WITH ERROR BARS (X = PARAPHRASER)
# ---------------------------------------------------------

plt.figure(figsize=(12, 6.5))

x = np.arange(len(paraphrasers))
labels = [p.replace("text_", "") for p in paraphrasers]

for ds in datasets:
    plt.errorbar(
        x,
        mean_f1[ds],
        yerr=std_f1[ds],
        marker="o",
        linewidth=2,
        capsize=4,
        label=ds.upper()
    )

plt.xticks(x, labels, rotation=30)
plt.ylabel("Mean BERTScore F1")
plt.title("Mean BERTScore F1 Across Paraphrasers (Lines = Datasets)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(title="Dataset", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

# ---------------------------------------------------------
# SAVE THE FIGURE
# ---------------------------------------------------------
os.makedirs("output", exist_ok=True)
plt.savefig("output/mean_f1_by_paraphraser.png", dpi=300, bbox_inches="tight")

plt.show()


### BERTScore F1 by Paraphraser Across Datasets (Using Grouped Bar Chart for Better Visualization)

In [0]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------

datasets = ["cmv", "eli5", "sci_gen", "tldr", "wp", "xsum", "yelp"]
paraphrasers = [
    "text_chatgpt",
    "text_palm",
    "text_dipper_low",
    "text_dipper",
    "text_dipper_high",
    "text_pegasus_slight",
    "text_pegasus_full",
]

cache_dir = "output"

# ---------------------------------------------------------
# LOAD AND AGGREGATE MEAN + STD F1
# ---------------------------------------------------------

mean_f1 = {ds: [] for ds in datasets}
std_f1  = {ds: [] for ds in datasets}

for ds in datasets:
    path = os.path.join(cache_dir, f"bertscore_{ds}_cache.pkl")
    with open(path, "rb") as f:
        bert_results = pickle.load(f)

    for p in paraphrasers:
        f1_vals = [triple[2] for triple in bert_results[p] if triple[2] is not None]
        mean_f1[ds].append(np.mean(f1_vals))
        std_f1[ds].append(np.std(f1_vals))

# ---------------------------------------------------------
# GROUPED BAR PLOT (X = PARAPHRASER, GROUPS = DATASETS)
# ---------------------------------------------------------

plt.figure(figsize=(14, 7))

x = np.arange(len(paraphrasers))
labels = [p.replace("text_", "") for p in paraphrasers]

num_datasets = len(datasets)
bar_width = 0.11

soft_colors = [
    "#7FC8C9",  # soft teal
    "#9ED9A0",  # light sea green
    "#F5C16C",  # warm amber
    "#F2E88B",  # soft gold
    "#A7D3E8",  # misty blue
    "#FF9AA2",  # soft coral-red
    "#9ED7D5",  # pale turquoise
]

for i, ds in enumerate(datasets):
    plt.bar(
        x + i * bar_width,
        mean_f1[ds],
        yerr=std_f1[ds],
        width=bar_width,
        capsize=3,
        color=soft_colors[i],
        label=ds.upper()
    )

plt.xticks(x + bar_width * (num_datasets - 1) / 2, labels, rotation=30)
plt.ylabel("Mean BERTScore F1")
plt.title("Mean BERTScore F1 Across Paraphrasers")
plt.ylim(0.8, 1.025)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(title="Dataset", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

os.makedirs("output", exist_ok=True)
plt.savefig("output/mean_f1_by_paraphraser_grouped.png", dpi=300, bbox_inches="tight")

plt.show()



In [0]:
# ---------------------------------------------------------
# UNIFIED BERTSCORE CACHE + HUMAN vs LLM-SOURCED DRIFT (F1 ONLY)
# ---------------------------------------------------------

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------

ALL_DATASETS = ["cmv", "eli5", "sci_gen", "tldr", "wp", "xsum", "yelp"]

paraphraser_cols = [
    ("chatgpt",        "text_chatgpt"),
    ("palm",           "text_palm"),
    ("dipper_low",     "text_dipper_low"),
    ("dipper_mid",     "text_dipper"),
    ("dipper_high",    "text_dipper_high"),
    ("pegasus_slight", "text_pegasus_slight"),
    ("pegasus_full",   "text_pegasus_full"),
]

paraphraser_order = [
    "chatgpt",
    "palm",
    "dipper_low",
    "dipper_mid",
    "dipper_high",
    "pegasus_slight",
    "pegasus_full",
]

# ---------------------------------------------------------
# LOAD ALL BERTSCORE CACHES + MERGE INTO ONE TABLE
# ---------------------------------------------------------

rows = []

for ds in ALL_DATASETS:
    cache_path = Path(f"output/bertscore_{ds}_cache.pkl")
    print(f"Loading {cache_path} ...")

    with open(cache_path, "rb") as f:
        cache = pickle.load(f)

    subset = paired_t1[paired_t1["dataset"] == ds].reset_index(drop=True)
    n_rows = len(subset)

    for p_name, p_col in paraphraser_cols:
        score_list = cache[p_col]

        # ---- FIX: trim to avoid out-of-bounds ----
        usable_len = min(len(score_list), n_rows)

        for i in range(usable_len):
            P, R, F1 = score_list[i]
            if F1 is None:
                continue

            row = subset.iloc[i]

            rows.append({
                "dataset": ds,
                "key": row["key"],
                "source": row["source"],
                "paraphraser": p_name,
                "bert_f1": F1,
            })

bert_df = pd.DataFrame(rows)
print("Unified BERTScore table shape:", bert_df.shape)

# ---------------------------------------------------------
# SAVE UNIFIED CACHE
# ---------------------------------------------------------

bert_df.to_pickle("output/bertscore_all_datasets.pkl")
print("Saved unified BERTScore cache → output/bertscore_all_datasets.pkl")

# ---------------------------------------------------------
# HUMAN vs LLM SOURCE GROUPING
# ---------------------------------------------------------

bert_df["source_group"] = bert_df["source"].apply(
    lambda s: "Human" if s == "Human" else "LLM"
)

bert_df["paraphraser"] = pd.Categorical(
    bert_df["paraphraser"],
    categories=paraphraser_order,
    ordered=True
)

# ---------------------------------------------------------
# COMPUTE MEAN + STD FOR HUMAN vs LLM (F1 ONLY)
# ---------------------------------------------------------

stats = (
    bert_df
    .groupby(["source_group", "paraphraser"])["bert_f1"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values(["paraphraser", "source_group"])
)

# ---------------------------------------------------------
# PLOT — BERTScore F1 (Human vs LLM)
# ---------------------------------------------------------

x = np.arange(len(paraphraser_order))
bar_width = 0.35

means_h = stats[stats["source_group"]=="Human"]["mean"].values
means_l = stats[stats["source_group"]=="LLM"]["mean"].values
std_h   = stats[stats["source_group"]=="Human"]["std"].values
std_l   = stats[stats["source_group"]=="LLM"]["std"].values

fig, ax = plt.subplots(figsize=(9,6))

ax.bar(x - bar_width/2, means_h, width=bar_width, label="Human", alpha=0.85)
ax.errorbar(x - bar_width/2, means_h, yerr=std_h, fmt="none", ecolor="black", capsize=4)

ax.bar(x + bar_width/2, means_l, width=bar_width, label="LLM", alpha=0.85)
ax.errorbar(x + bar_width/2, means_l, yerr=std_l, fmt="none", ecolor="black", capsize=4)

ax.set_title("BERTScore F1 — Human vs LLM Sources", fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(paraphraser_order, rotation=45, ha="right", fontsize=12)

ax.set_ylim(0.8, 1.02)
ax.set_yticks(np.arange(0.775, 1.101, 0.05))

ax.set_ylabel("BERTScore F1", fontsize=14)
ax.legend(title="Source Group", fontsize=11, title_fontsize=12)

plt.tight_layout()
plt.savefig("output/bertscore_f1_human_vs_llm.png", dpi=300, bbox_inches="tight")
plt.show()


### Diagnostic

In [0]:
for ds in ALL_DATASETS:
    subset = paired_t1[paired_t1["dataset"] == ds]
    print(f"\nDataset {ds}: subset rows = {len(subset)}")

    cache_path = f"output/bertscore_{ds}_cache.pkl"
    with open(cache_path, "rb") as f:
        cache = pickle.load(f)

    for p_name, p_col in paraphraser_cols:
        print(f"  {p_name:15s} cache len = {len(cache[p_col])}")
